In [1]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import src.audio_transforms as at
import torch
import yaml

import argparse
from argparse import ArgumentParser
from corpus.swc_mono_test import SWCMonoTestSet
from corpus.speaker_room_dataset import SpeakerRoomDataset


from tqdm.auto import tqdm
from datetime import datetime
import sys 

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

### Get most recent config
# config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
# ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=2-step=32288.ckpt"

config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
# ckpt_path = "attn_cue_models/word_task_standard_v08/checkpoints/epoch=3-step=51756-v1.ckpt"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"


In [4]:

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 2
config['hparas']['batch_size'] = 8 # config['data']['loader']['batch_size'] // args.gpus
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
model_in_sr = config['audio']['rep_kwargs']['sr']

In [5]:
model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).cuda()
model = model.eval()
coch_gram = model.coch_gram.cuda()

num_classes={'num_words': 800}


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [18]:

diotic_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                #     at.TimeReverseBackgroundSignal(time_dim=[-1]),
                    at.CombineWithRandomDBSNR(low_snr=-15, high_snr=-15), 
                    at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                    at.DuplicateChannel(),

            ])


diotic_transforms = diotic_transforms.cuda()



In [19]:
dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
                            excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
                            cue_type='voice_and_location',
                            sr=model_in_sr) 

def single_signal_collate_fn(batch):
    #apply transforsms to batch
    cues = torch.stack([diotic_transforms(cue, None)[0] for cue, fg, bg, label, confusion in batch])
    mixtures = torch.stack([diotic_transforms(fg, bg)[0] for cue, fg, bg, label, confusion in batch]).type(torch.FloatTensor)
    labels = torch.tensor([label for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    confusion = torch.tensor([confusion for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    return cues, mixtures, labels, confusion


dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, num_workers=config['num_workers'], collate_fn=single_signal_collate_fn)


In [20]:
# stim_path = "/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/"
# dataset = SWCMonoTestSet(stim_path=stim_path,
#                             cond_ix=33, # 4 for music at 3 db, 40 for clean speech  33 for 1-distrctor at 0dB
#                             model_sr=44100,
#                             label_type='CV')

# audio_transforms = at.AudioCompose([
#                     at.AudioToTensor(),
#                     # at.CombineWithRandomDBSNR(low_snr=snr, high_snr=snr), 
#                     at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
#                     at.UnsqueezeAudio(dim=0),
#                     at.DuplicateChannel()
#             ])  
# def collate_fn(batch):
#     #apply transforsms to batch
#     cues = torch.stack([audio_transforms(cue, None)[0] for cue, _, _ in batch])
#     mixtures = torch.stack([audio_transforms(mix, None)[0] for _, mix,  _ in batch])
#     labels = torch.tensor([label for _, _, label in batch]).type(torch.LongTensor)
#     return cues, mixtures, labels

# dataloader = torch.utils.data.DataLoader(dataset,
#                                              batch_size=16,
#                                              shuffle=False,
#                                              collate_fn=collate_fn,
#                                              num_workers=0)

### Try diotically first, then spatialize 

In [21]:
output_dict = {'results': None, 'confusions': None}
accuracies = []
pred_list = []
true_word_int = []
confusions_list  = []
all_probs_of_interest = []
guessed_both = []
ix = 0 
with torch.no_grad(): 
    for batch in tqdm(dataloader):
        # if ix > 10:
        #     break
        cue, mixture, label, confusion = batch

        cue, mixture = coch_gram(cue.cuda(), mixture.cuda())
        # cue_mask_ixs = torch.arange(cue.shape[0])
        logits = model(cue, mixture, None)
        probs = logits.softmax(dim=-1).cpu().detach().numpy()

        # get top 2 probs for each example
        top_2_probs = torch.topk(logits.softmax(-1), 5, dim=-1).indices.cpu().detach().numpy()
        return_both = (np.isin(label.numpy(), top_2_probs) * np.isin(confusion.numpy(), top_2_probs)).astype('int')
        
        targ_probs = probs[torch.arange(probs.shape[0]), label]
        conf_probs = probs[torch.arange(probs.shape[0]), confusion]
        probs_of_interest = np.concatenate([targ_probs[:, None], conf_probs[:, None]], axis=1)

        preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
        true_word = label.numpy().astype('int')
        accuracy = (preds == true_word).astype('int')
        conf = (preds == confusion.numpy().astype('int')).astype('int')
        confusions_list.append(conf)
        accuracies.append(accuracy)
        pred_list.append(preds)
        true_word_int.append(true_word)
        all_probs_of_interest.append(probs_of_interest)
        guessed_both.append(return_both)
        ix += 1 
        if ix == 10:
            break
        
accuracies = np.concatenate(accuracies)
pred_list = np.concatenate(pred_list)
true_word_int = np.concatenate(true_word_int)
confusions_list = np.concatenate(confusions_list)
all_probs_of_interest = np.concatenate(all_probs_of_interest, axis=0)
guessed_both = np.concatenate(guessed_both)

output_dict['probs_of_interest'] = all_probs_of_interest
output_dict['results'] = accuracies
output_dict['preds'] = pred_list
output_dict['true_word_int'] = true_word_int

print(f"Accuracy using time-reversed distractor: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
print(f"Confusions using time-reversed distractor: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")



  9%|▉         | 9/99 [00:46<07:43,  5.15s/it]

Accuracy using time-reversed distractor: 0.16, (0.02)
Confusions using time-reversed distractor: 0.39, (0.03)
Guessed both talker words: 0.16, (0.02)


In [ ]:
# at -6 db

# Accuracy using time-reversed distractor: 0.38, (0.03)
# Confusions using time-reversed distractor: 0.00, (0.00)
# Guessed both talker words: 0.09, (0.02) 



In [25]:
print(f"Accuracy using time-reversed distractor: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
print(f"Confusions using time-reversed distractor: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")

Accuracy using time-reversed distractor: 0.57, (0.03)
Confusions using time-reversed distractor: 0.00, (0.00)
